# 05 — Batch like-games experiments

Reads the single top-level `experiments` list in `configs/experiments.yaml`. Experiments with identical similarity settings share one saved neighbor selection. Each experiment’s `forecast` block is applied in 06. Only pending work is generated. Completed work is verified and skipped across full reruns and cluster restarts; failed or interrupted attempts retain their experiment identity.

## First use and reset

1. Put `experiments.yaml` inside a `configs` folder beside this notebook, or set its absolute `/Workspace/...` path in cell 2.
2. For existing batch-format tables the revised experiments append with new IDs. Legacy pre-ledger tables are detected during preflight.
3. Review the enabled experiments in the YAML and run the notebook in order.
4. Run one manual batch at a time. A retry assumes any previous writer has stopped.

Normal processing has no overwrite/reset switch. Legacy results without the new status records stop the batch with a reset instruction. A reset is a one-time migration action; it is not part of adding experiments.

## Configuration and source snapshots

- The YAML is the only configuration source. Experiment overrides merge into shared defaults, including individual feature weights and neighbor limits. Unknown/duplicate YAML keys and duplicate enabled settings are rejected before saving.
- `data.revision` names a fixed AI source snapshot. A null `source_version` resolves the latest version when this revision is first registered, then reuses the saved version for later batches. To use updated AI data, explicitly change `data.revision` (and optionally `source_version`).
- Identity depends on the data revision/snapshot, similarity settings, prediction-date fallback, and implementation version. Names, enabled flags, batch membership, YAML location, and monitoring options do not create new experiments. An existing identity keeps its original readable name; the batch summary also shows the requested name.
- If you intentionally change the implementation, increment `SIMILARITY_IMPLEMENTATION_VERSION` in the configuration helper cell before running a new generation.

## Target and neighbor rules

Every valid canonical game key is retained as a target, whether or not it has Sales, Rental, or Eilers matches. Actuals are not required here. Evaluation eligibility belongs to updated notebook 07, per channel and metric.

Every game gets up to five overall content comparisons against other catalog games, including later releases. These appear in `overall_similar_games` for review. The three forecast lists use separately scaled, date-eligible candidates.

Notebook 05 selects the final neighbors for each forecast. A sales neighbor needs 6 distinct observed sales months by default. A rental neighbor needs 9 distinct rental months. Both also need a usable Eilers observation. Each date must precede the target prediction month. Observed zeros count, gaps are allowed, repeated rows count once per calendar month, and filled months do not count. These checks apply before ranking, including to a preferred P3 parent. A target needs no commercial history of its own.

05 ranks eligible games, excludes the target's own mapping, deduplicates mapped source names, and applies each channel's shortlist limit. It then applies the distance cutoff and nearest-eligible fallback. The final ordered arrays are `sales_similar_games`, `rental_similar_games`, and `eilers_similar_games`. Commercial threshold dates for selected games are saved in `neighbor_history_threshold_months`. The Eilers-only list uses its existing performance rules.

Use the shared project YAML. The defaults can be overridden with `sales_min_observed_months` and `rental_min_observed_months` under `forecast.settings`. 05 reads the commercial and Eilers Delta snapshots pinned to `forecast.data_revision` and includes selection settings and source versions in each generation's identity.

A missing release date stays null. Its prediction date comes from the YAML override or from the UTC date of the first updated batch for the AI revision and snapshot. The assumed date is saved and reused on retries. Undated targets bypass the minimum age gap. An undated neighbor is treated as recent and can be considered from the assumed date onward. 05 checks commercial neighbor history as of the prediction month. Known-date pairs continue to use the configured release cutoff and age gap, with the existing P3 parent date exception.

## Monitoring and tracking

- `similarity_runs`: durable processing/tracking status, identity, source snapshot, configuration, output count/hash, coverage, and last error.
- `similarity_run_attempts`: persistent attempt history, including failures/interruption and retries.
- `like_games_ai_attr`: saved neighbor generations with identity and MLflow lineage.

The final cells show the current batch, recent generations, neighbor coverage, date failures, and attempts. These are similarity/neighbor coverage counts, not accuracy metrics or proof of ground-truth availability. No scheduler is required.

Delta completion is authoritative and is published only after result verification. MLflow runs under `/Shared/AI_Feature_Extraction_Similarity` record settings, YAML, preprocessing, prepared input, and coverage. A tracking outage is recorded separately; a rerun retries logging without recalculating neighbors.

## Downstream migration

Use the companion batch versions of notebooks 06 and 07 for this workflow. Existing columns are retained and new lineage/date fields are additive. The shared YAML supplies the source definitions and selection settings used by both 05 and 06. Game charts live in the separate `Review_Forecasts` notebook.

Local behavioral checks cover YAML identity, candidate/date/parent rules, no-match targets, hashes, and retry state. Spark/Delta/MLflow integration must be exercised in Databricks.


In [0]:
# The only notebook input: location of the YAML configuration file.
# Prefer an absolute /Workspace/... path if your working directory differs.
CONFIG_PATH = "configs/experiments.yaml"


## Configuration validation and identity

Helper definitions only; no tables are changed by this cell.


In [0]:
"""Validated YAML configuration and semantic work identities for notebook 05."""
import copy
import hashlib
import json
import math
import re
from datetime import date, datetime
from numbers import Integral, Real
from pathlib import Path

SIMILARITY_IMPLEMENTATION_VERSION = "05_history_at_selection_v3"
WEIGHT_GROUPS = ("mechanics", "themes", "reels", "persistence", "ways", "months_since_release")
PARENT_PROCESSES = {"p1_parent_games_only", "p2_current", "p3_include_parent"}


def config_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False,
                      default=lambda x: x.isoformat() if isinstance(x, (date, datetime)) else str(x))


def config_sha256(value):
    return hashlib.sha256(config_json(value).encode("utf-8")).hexdigest()


def quoted_table(name):
    return ".".join("`" + p.replace("`", "``") + "`" for p in name.split("."))


def _keys(value, allowed, where, required=()):
    if not isinstance(value, dict):
        raise ValueError(f"{where} must be a mapping.")
    unknown, missing = set(value) - set(allowed), set(required) - set(value)
    if unknown or missing:
        raise ValueError(f"{where}: unknown keys={sorted(unknown)}, missing keys={sorted(missing)}")


def _text(value, where):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{where} must be a nonempty string.")
    return value.strip()


def _integer(value, where, minimum=0):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{where} must be an integer >= {minimum}.")
    return int(value)


def _date(value, where):
    if value is None:
        return None
    if isinstance(value, datetime):
        value = value.date()
    if isinstance(value, date):
        return value.isoformat()
    try:
        return date.fromisoformat(str(value)).isoformat()
    except ValueError as exc:
        raise ValueError(f"{where} must be YYYY-MM-DD or null.") from exc


def _table(value, where):
    value = _text(value, where)
    if len(value.split(".")) != 3 or not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in value.split(".")):
        raise ValueError(f"{where} must be a three-part catalog.schema.table name.")
    return value


def validate_similarity_settings(settings):
    allowed = {"raw_weights", "distance_method", "parent_game_process", "neighbor_limits",
               "neighbor_released_after", "minimum_age_gap_months"}
    _keys(settings, allowed, "similarity settings", allowed)
    settings = copy.deepcopy(settings)
    settings["distance_method"] = _text(settings["distance_method"], "distance_method").lower()
    if settings["distance_method"] not in {"cosine", "euclidean"}:
        raise ValueError("distance_method must be cosine or euclidean.")
    if settings["parent_game_process"] not in PARENT_PROCESSES:
        raise ValueError(f"Unknown parent_game_process: {settings['parent_game_process']!r}")
    _keys(settings["raw_weights"], WEIGHT_GROUPS, "raw_weights", WEIGHT_GROUPS)
    for key, value in settings["raw_weights"].items():
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or value < 0:
            raise ValueError(f"raw_weights.{key} must be a finite nonnegative number.")
        settings["raw_weights"][key] = float(value)
    if not any(settings["raw_weights"].values()):
        raise ValueError("At least one feature-group weight must be positive.")
    _keys(settings["neighbor_limits"], {"sales", "rental", "eilers"}, "neighbor_limits", {"sales", "rental", "eilers"})
    settings["neighbor_limits"] = {k: _integer(v, f"neighbor_limits.{k}") for k, v in settings["neighbor_limits"].items()}
    settings["minimum_age_gap_months"] = _integer(settings["minimum_age_gap_months"], "minimum_age_gap_months")
    settings["neighbor_released_after"] = _date(settings["neighbor_released_after"], "neighbor_released_after")
    if settings["neighbor_released_after"] is None:
        raise ValueError("neighbor_released_after must be a date.")
    return settings


def named_forecast_settings(item, similarity):
    """Weights are ordered by historical demand in each month, highest first."""
    name = item["name"]
    spec = item.get("forecast", {})
    _keys(spec, {"rank_weights", "train_bias_correction"}, f"{name}.forecast")
    weights = spec.get("rank_weights", [1.0])
    if not isinstance(weights, list) or not weights:
        raise ValueError(f"{name}.forecast.rank_weights must be a nonempty list.")
    if any(isinstance(w, bool) or not isinstance(w, Real) or not math.isfinite(w) or w <= 0 for w in weights):
        raise ValueError(f"{name}.forecast.rank_weights must contain positive finite numbers.")
    if any(a < b for a, b in zip(weights, weights[1:])):
        raise ValueError(f"{name}.forecast.rank_weights must be ordered highest weight first.")
    if not math.isfinite(sum(weights)):
        raise ValueError(f"{name}.forecast.rank_weights sum must be finite.")
    correct = spec.get("train_bias_correction", False)
    if not isinstance(correct, bool):
        raise ValueError(f"{name}.forecast.train_bias_correction must be true or false.")
    return {"name": name, "rank_weights": [float(w) for w in weights],
            "train_bias_correction": correct, "similarity_settings_hash": config_sha256(similarity)}


def load_experiment_config(config_path, require_enabled=True):
    try:
        import yaml
    except ImportError as exc:
        raise ImportError("PyYAML is required. Install PyYAML on your Databricks environment, then rerun.") from exc

    # Reject duplicate YAML keys rather than silently replacing a weight/setting.
    class UniqueLoader(yaml.SafeLoader):
        pass
    def unique_mapping(loader, node, deep=False):
        result = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node, deep=deep)
            if key in result:
                raise ValueError(f"Duplicate YAML key {key!r} on line {key_node.start_mark.line + 1}.")
            result[key] = loader.construct_object(value_node, deep=deep)
        return result
    UniqueLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    path = Path(config_path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Configuration not found: {path}. Put experiments.yaml in configs beside this notebook, or set CONFIG_PATH to its absolute /Workspace/... path.")
    original_text = path.read_text(encoding="utf-8")
    raw = yaml.load(original_text, Loader=UniqueLoader)
    sections = {"schema_version", "data", "outputs", "tracking", "monitoring", "similarity_defaults", "experiments"}
    # Downstream stages share this file without changing 05's semantic inputs.
    _keys(raw, sections | {"forecast", "evaluation"}, "configuration", sections)
    if isinstance(raw["schema_version"], bool) or raw["schema_version"] != 1:
        raise ValueError("Only schema_version: 1 is supported.")
    _keys(raw["data"], {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"}, "data",
          {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"})
    data = copy.deepcopy(raw["data"])
    data["revision"] = _text(data["revision"], "data.revision")
    data["ai_attributes_table"] = _table(data["ai_attributes_table"], "data.ai_attributes_table")
    if data["source_version"] is not None:
        data["source_version"] = _integer(data["source_version"], "data.source_version")
    data["prediction_as_of_date"] = _date(data["prediction_as_of_date"], "data.prediction_as_of_date")
    output_keys = {"like_games_table", "runs_table", "attempts_table"}
    _keys(raw["outputs"], output_keys, "outputs", output_keys)
    outputs = {k: _table(v, f"outputs.{k}") for k, v in raw["outputs"].items()}
    if len(set(outputs.values())) != len(outputs) or data["ai_attributes_table"] in outputs.values():
        raise ValueError("Output tables must be distinct and cannot be the AI source table.")
    _keys(raw["tracking"], {"experiment_path"}, "tracking", {"experiment_path"})
    tracking_path = _text(raw["tracking"]["experiment_path"], "tracking.experiment_path")
    if not tracking_path.startswith("/"):
        raise ValueError("tracking.experiment_path must start with /.")
    _keys(raw["monitoring"], {"history_rows"}, "monitoring", {"history_rows"})
    history_rows = _integer(raw["monitoring"]["history_rows"], "monitoring.history_rows", 1)
    defaults = validate_similarity_settings(raw["similarity_defaults"])
    if not isinstance(raw["experiments"], list) or not raw["experiments"]:
        raise ValueError("experiments must be a nonempty list.")
    experiments, names = [], set()
    for index, item in enumerate(raw["experiments"]):
        _keys(item, {"name", "enabled", "overrides", "forecast"}, f"experiments[{index}]", {"name"})
        name = _text(item["name"], f"experiments[{index}].name")
        if name in names:
            raise ValueError(f"Duplicate experiment name: {name!r}")
        names.add(name)
        enabled = item.get("enabled", True)
        if not isinstance(enabled, bool):
            raise ValueError(f"{name}.enabled must be true or false.")
        overrides = item.get("overrides", {})
        _keys(overrides, defaults, f"{name}.overrides")
        merged = copy.deepcopy(defaults)
        for key, value in overrides.items():
            if key in {"raw_weights", "neighbor_limits"}:
                _keys(value, defaults[key], f"{name}.overrides.{key}")
                merged[key].update(copy.deepcopy(value))
            else:
                merged[key] = copy.deepcopy(value)
        merged = validate_similarity_settings(merged)
        forecast = named_forecast_settings(item, merged)
        experiments.append({"name": name, "enabled": enabled, "settings": merged, "forecast": forecast})
    if require_enabled and not any(x["enabled"] for x in experiments):
        raise ValueError("Enable at least one experiment in the YAML file.")
    return {"data": data, "outputs": outputs, "tracking": {"experiment_path": tracking_path},
            "monitoring": {"history_rows": history_rows}, "experiments": experiments,
            "config_path": str(path), "config_text": original_text,
            "yaml_sha256": hashlib.sha256(original_text.encode("utf-8")).hexdigest()}


def similarity_work_identity(experiment, data, source_version, selection_history):
    semantic = {"implementation_version": SIMILARITY_IMPLEMENTATION_VERSION,
                "data_revision": data["revision"], "source_table": data["ai_attributes_table"],
                "source_table_version": int(source_version),
                "prediction_as_of_date": data["prediction_as_of_date"],
                "prediction_as_of_source": data.get("prediction_as_of_source", "configured"),
                "similarity": copy.deepcopy(experiment["settings"]),
                "selection_history": copy.deepcopy(selection_history)}
    digest = config_sha256(semantic)
    return digest, semantic


def preflight_similarity_tables(spark, config):
    """Refuse to mix pre-ledger legacy generations with the new history."""
    results, runs = (config["outputs"][x] for x in ["like_games_table", "runs_table"])
    if spark.catalog.tableExists(results):
        frame = spark.table(results)
        if frame.limit(1).count():
            if "similarity_work_key" not in frame.columns or not spark.catalog.tableExists(runs):
                raise ValueError("Previous similarity results use the old format. Run 00_Reset_Similarity_Pipeline once before the first new batch. No results were overwritten.")
            ledger = spark.table(runs)
            if not {"work_key", "experiment_id", "metadata_json"}.issubset(ledger.columns):
                raise ValueError("The similarity status table has an incompatible schema. Run the one-time reset before starting.")
            if frame.select("experiment_id").distinct().join(ledger.select("experiment_id").distinct(), "experiment_id", "left_anti").limit(1).count():
                raise ValueError("Similarity rows without a matching status record were found. Resolve/reset them before running a new batch.")


def resolve_similarity_source_version(spark, config):
    """A data revision pins one source snapshot across later manually run batches."""
    from pyspark.sql import functions as F
    data, runs = config["data"], config["outputs"]["runs_table"]
    pinned = []
    if spark.catalog.tableExists(runs):
        ledger = spark.table(runs)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Incompatible similarity status table; perform the one-time reset first.")
        pinned = [int(r["source_table_version"]) for r in ledger.where(
            (F.get_json_object("metadata_json", "$.data_revision") == data["revision"])
            & (F.col("source_path") == data["ai_attributes_table"])
        ).select("source_table_version").distinct().collect()]
    if len(set(pinned)) > 1:
        raise ValueError("This data revision has multiple source snapshots. Choose a new revision after resolving the history.")
    if pinned:
        version = pinned[0]
        if data["source_version"] is not None and data["source_version"] != version:
            raise ValueError(f"Revision {data['revision']!r} is already pinned to Delta version {version}. Change data.revision to use another version.")
        return version
    if data["source_version"] is not None:
        return data["source_version"]
    return int(spark.sql(f"DESCRIBE HISTORY {quoted_table(data['ai_attributes_table'])}").agg(F.max("version").alias("version")).first()["version"])


FORECAST_SOURCE_KEYS = {"sales", "lease", "performance"}
FORECAST_OUTPUT_KEYS = {"demand", "performance", "neighbors", "runs", "attempts"}
DEFAULT_HISTORY_MONTHS = {"sales_min_observed_months": 6, "rental_min_observed_months": 9}
FORECAST_SETTING_KEYS = {"distance_cutoff", "minimum_neighbors", "horizon_months", "slot_counts",
                         "lease_value_column", "lease_metric_name", "lease_curve_label", "performance_metrics"}


def load_forecast_config(config_path):
    # Both stages use the same source definitions and selection options.
    common = load_experiment_config(config_path, require_enabled=False)
    import yaml
    raw = yaml.safe_load(common["config_text"])
    forecast = raw.get("forecast")
    forecast_keys = {"data_revision", "sources", "source_versions", "settings", "outputs", "tracking", "selection", "monitoring"}
    if isinstance(forecast, dict) and "experiments" in forecast:
        raise ValueError("Move forecast.experiments into the named entries in the top-level experiments list.")
    _keys(forecast, forecast_keys, "forecast", forecast_keys)
    revision = _text(forecast["data_revision"], "forecast.data_revision")
    _keys(forecast["sources"], FORECAST_SOURCE_KEYS, "forecast.sources", FORECAST_SOURCE_KEYS)
    sources = {k: _table(v, f"forecast.sources.{k}") for k, v in forecast["sources"].items()}
    _keys(forecast["source_versions"], FORECAST_SOURCE_KEYS, "forecast.source_versions", FORECAST_SOURCE_KEYS)
    versions = {k: None if v is None else _integer(v, f"forecast.source_versions.{k}")
                for k, v in forecast["source_versions"].items()}
    _keys(forecast["outputs"], FORECAST_OUTPUT_KEYS, "forecast.outputs", FORECAST_OUTPUT_KEYS)
    outputs = {k: _table(v, f"forecast.outputs.{k}") for k, v in forecast["outputs"].items()}
    protected_inputs = {common["data"]["ai_attributes_table"], *sources.values(), *common["outputs"].values()}
    if len(set(outputs.values())) != len(outputs) or set(outputs.values()) & protected_inputs:
        raise ValueError("Forecast output tables must be distinct and cannot replace any source or similarity tables.")
    _keys(forecast["settings"], FORECAST_SETTING_KEYS | set(DEFAULT_HISTORY_MONTHS), "forecast.settings", FORECAST_SETTING_KEYS)
    settings = copy.deepcopy(forecast["settings"])
    for name, default in DEFAULT_HISTORY_MONTHS.items():
        settings[name] = _integer(settings.get(name, default), f"forecast.settings.{name}", 1)
    cutoff = settings["distance_cutoff"]
    if isinstance(cutoff, bool) or not isinstance(cutoff, Real) or not math.isfinite(float(cutoff)) or cutoff < 0:
        raise ValueError("forecast.settings.distance_cutoff must be finite and >= 0.")
    settings["distance_cutoff"] = float(cutoff)
    settings["minimum_neighbors"] = _integer(settings["minimum_neighbors"], "minimum_neighbors")
    settings["horizon_months"] = _integer(settings["horizon_months"], "horizon_months", 1)
    slots = settings["slot_counts"]
    if not isinstance(slots, list) or any(isinstance(x, bool) or not isinstance(x, Real) or not math.isfinite(float(x)) or x <= 0 for x in slots):
        raise ValueError("slot_counts must be a list of finite positive numbers; [] means all positive counts.")
    if len(set(slots)) != len(slots):
        raise ValueError("slot_counts cannot contain duplicates.")
    settings["slot_counts"] = sorted(float(x) for x in slots)
    for key in ["lease_value_column", "lease_metric_name", "lease_curve_label"]:
        settings[key] = _text(settings[key], key)
    if not isinstance(settings["performance_metrics"], list) or not settings["performance_metrics"]:
        raise ValueError("performance_metrics must be a nonempty list.")
    settings["performance_metrics"] = [_text(x, "performance_metrics entry") for x in settings["performance_metrics"]]
    if len(set(settings["performance_metrics"])) != len(settings["performance_metrics"]):
        raise ValueError("performance_metrics cannot contain duplicates.")
    settings["performance_metrics"] = sorted(settings["performance_metrics"])
    _keys(forecast["tracking"], {"experiment_path"}, "forecast.tracking", {"experiment_path"})
    mlflow_path = _text(forecast["tracking"]["experiment_path"], "forecast.tracking.experiment_path")
    if not mlflow_path.startswith("/"):
        raise ValueError("forecast.tracking.experiment_path must be an absolute workspace path.")
    _keys(forecast["selection"], {"similarity_experiment_ids"}, "forecast.selection", {"similarity_experiment_ids"})
    ids = forecast["selection"]["similarity_experiment_ids"]
    if not isinstance(ids, list):
        raise ValueError("similarity_experiment_ids must be a list; [] discovers all completed 05 experiments.")
    ids = [_text(x, "similarity_experiment_ids entry") for x in ids]
    if len(set(ids)) != len(ids):
        raise ValueError("similarity_experiment_ids cannot contain duplicates.")
    _keys(forecast["monitoring"], {"history_rows"}, "forecast.monitoring", {"history_rows"})
    return {"common": common, "data_revision": revision, "sources": sources, "source_versions": versions,
            "settings": settings, "outputs": outputs, "tracking": {"experiment_path": mlflow_path},
            "selection": {"similarity_experiment_ids": ids},
            "monitoring": {"history_rows": _integer(forecast["monitoring"]["history_rows"], "forecast.monitoring.history_rows", 1)}}


def neighbor_selection_policy(config, versions):
    keys = ("sales_min_observed_months", "rental_min_observed_months", "distance_cutoff",
            "minimum_neighbors", "lease_value_column", "performance_metrics", "slot_counts")
    return {
        "data_revision": config["data_revision"],
        "source_paths": dict(config["sources"]),
        "source_versions": dict(versions),
        "settings": {key: config["settings"][key] for key in keys},
    }


def resolve_forecast_source_versions(spark, config):
    from pyspark.sql import functions as F
    table = config["outputs"]["runs"]
    matches = []
    if spark.catalog.tableExists(table):
        ledger = spark.table(table)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Legacy forecast manifest found. Reset scope '06_07' before using new 06; preserve 05.")
        matches = ledger.where(F.get_json_object("metadata_json", "$.data_revision") == config["data_revision"]).select(
            "source_versions_json", "config_json").distinct().collect()
    versions = None
    for row in matches:
        previous_config = json.loads(row["config_json"])
        if previous_config["source_paths"] != config["sources"]:
            raise ValueError("This forecast data revision is already assigned to other source tables. Choose another forecast.data_revision.")
        candidate = {k: int(v) for k, v in json.loads(row["source_versions_json"]).items()}
        if versions is not None and candidate != versions:
            raise ValueError("This forecast revision contains conflicting source snapshots; resolve its history before running.")
        versions = candidate
    # A completed or interrupted 05 run can be the first writer for this revision.
    similarity_runs = config["common"]["outputs"]["runs_table"]
    if spark.catalog.tableExists(similarity_runs):
        selections = (spark.table(similarity_runs)
                      .where(F.get_json_object("config_json", "$.selection_history.data_revision") == config["data_revision"])
                      .select(F.get_json_object("config_json", "$.selection_history").alias("history")).distinct())
        for row in selections.collect():
            history = json.loads(row["history"])
            if history["source_paths"] != config["sources"]:
                raise ValueError("This data revision is pinned to different selection sources.")
            candidate = history["source_versions"]
            if versions is not None and versions != candidate:
                raise ValueError("Neighbor selection and forecast ledgers disagree on this revision's snapshots.")
            versions = candidate
    if versions is not None:
        for key, requested in config["source_versions"].items():
            if requested is not None and requested != versions[key]:
                raise ValueError(f"Revision is pinned to {key} version {versions[key]}; change forecast.data_revision to refresh.")
        return versions
    return {key: int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(config['sources'][key])}").agg(F.max("version").alias("v")).first()["v"])
        for key, requested in config["source_versions"].items()}


## Shared feature preparation and similarity rules

Reusable functions; feature preparation runs once for the pending batch.


In [0]:
"""Reusable, side-effect-free similarity generation for notebook 05.

Spark is used only to project the source's nested features. The deduplicated
feature population is collected once and shared by every experiment in a batch.
"""
import copy
import hashlib
import json
import math
from collections import Counter
from datetime import datetime, timezone
from numbers import Integral, Real

import numpy as np
import pandas as pd


def normalize_game_reference(value):
    if value is None or pd.isna(value):
        return None
    key = " ".join(str(value).strip().lower().split())
    return None if key in {"", "none", "null", "<na>", "nan", "na", "n/a", "__unknown__"} else key


def as_bool(value):
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def optional_string(record, column):
    value = record.get(column)
    return None if value is None or pd.isna(value) else str(value)


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False,
                      default=lambda item: item.isoformat())


def _numeric_feature_expr(source_columns, root_col, preferred_names):
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, NumericType, StringType

    def extract(expression):
        value = F.regexp_extract(expression.cast("string"), r"(-?\d+(?:\.\d+)?)", 1)
        return F.when(F.length(value) > 0, value.cast("double"))

    def leaves(data_type, prefix=()):
        if isinstance(data_type, StructType):
            return [leaf for field in data_type.fields
                    for leaf in leaves(field.dataType, prefix + (field.name,))]
        return [(prefix, data_type)]

    data_type = source_columns[root_col].dataType
    if isinstance(data_type, NumericType):
        return F.col(root_col).cast("double")
    if isinstance(data_type, StringType):
        return extract(F.col(root_col))
    if isinstance(data_type, StructType):
        fields = leaves(data_type)
        chosen = None
        for preferred in preferred_names:
            chosen = next(((path, typ) for path, typ in fields
                           if path[-1].lower() == preferred
                           and isinstance(typ, (NumericType, StringType))), None)
            if chosen is not None:
                break
        if chosen is None:
            chosen = next(((path, typ) for path, typ in fields if isinstance(typ, NumericType)), None)
        if chosen is None:
            chosen = next(((path, typ) for path, typ in fields if isinstance(typ, StringType)), None)
        if chosen is not None:
            path, leaf_type = chosen
            expression = F.col(root_col)
            for part in path:
                expression = expression.getField(part)
            return expression.cast("double") if isinstance(leaf_type, NumericType) else extract(expression)
    raise ValueError(f"Cannot extract a numeric feature from {root_col}: {data_type}")


def prepare_games(source_df):
    """Project numeric features and deduplicate the pinned Spark source once.

    Actual/match availability affects candidate pools, never target retention.
    The returned population is independent of experiment weights and settings.
    """
    from pyspark.sql import functions as F

    columns = {field.name: field for field in source_df.schema.fields}
    missing = {"game_name", "join_game_name_key"} - set(columns)
    if missing:
        raise ValueError(f"AI attributes missing canonical columns: {sorted(missing)}")
    themes = sorted(c for c in columns if c.startswith("theme_") and c.endswith("_score")
                    and c != "theme_presence_score")
    mechanics = sorted(c for c in columns if c.startswith("mechanic_") and c.endswith("_present"))
    mechanics = mechanics or sorted(c for c in columns if c.startswith("mechanic_") and c.endswith("_score"))
    persistence = [c for c in ["persistence_score", "perceived_persistence_score",
                              "persistence", "perceived_persistence"] if c in columns]
    reels = next((c for c in ["number_of_reels", "reels"] if c in columns), None)
    ways = next((c for c in ["ways", "lines_or_ways", "paylines_or_ways_count", "lines"] if c in columns), None)
    if not themes or not mechanics or not persistence or reels is None or ways is None:
        raise ValueError("Expected theme, mechanic, persistence, reels, and ways/lines feature columns.")
    own_column = "own_status" if "own_status" in columns else "owned_status" if "owned_status" in columns else None
    projected = [F.col("game_name"), F.col("join_game_name_key")]
    metadata = ["parent_game", "game_category", "parent_own_status", "game_classification",
                "game_release_date", "max_months_since_release",
                "to_sales_name_match", "to_rental_name_match"]
    for name in metadata:
        projected.append(F.col(name) if name in columns else F.lit(None).cast("string").alias(name))
    projected.append(F.col(own_column).alias("own_status") if own_column else F.lit(None).cast("string").alias("own_status"))
    # Preserve the previous Eilers fallback when a separate mapping column is absent.
    projected.append((F.col("to_eilers_ts_name_match") if "to_eilers_ts_name_match" in columns
                      else F.col("game_name")).alias("to_eilers_ts_name_match"))
    for flag, mapping in [("has_sales_match", "to_sales_name_match"),
                          ("has_rental_match", "to_rental_name_match"),
                          ("has_eilers_ts_match", "to_eilers_ts_name_match")]:
        if flag in columns:
            projected.append(F.col(flag))
        elif mapping in columns:
            mapped = F.lower(F.trim(F.col(mapping).cast("string")))
            projected.append((mapped.isNotNull() & ~mapped.isin("", "none", "null", "<na>", "nan", "na", "n/a", "__unknown__")).alias(flag))
        else:
            projected.append(F.lit(False).alias(flag))
    projected.extend([
        _numeric_feature_expr(columns, reels, ["number_of_reels", "reel_count", "count", "base_reels", "reels", "value", "score"]).alias("reels_clean"),
        _numeric_feature_expr(columns, ways, ["value", "ways", "lines", "count", "paylines_or_ways_count", "score"]).alias("ways_clean"),
    ])
    projected.extend(_numeric_feature_expr(columns, c, ["value", "score", "persistence", "persistence_score"]).alias(c)
                     for c in persistence)
    projected.extend(F.expr("try_cast(`" + c.replace("`", "``") + "` AS DOUBLE)").alias(c)
                     for c in themes + mechanics)
    groups = {"mechanics": mechanics, "themes": themes, "reels": ["reels_clean"],
              "persistence": persistence, "ways": ["ways_clean"],
              "months_since_release": ["max_months_since_release"]}
    return prepare_game_frame(source_df.select(*projected).toPandas(), groups)


def prepare_game_frame(frame, feature_groups):
    """Pure pandas preparation after Spark feature projection; useful for tests."""
    frame = frame.copy(deep=True)
    before_count = len(frame)
    frame["join_game_name_key"] = frame["join_game_name_key"].map(normalize_game_reference)
    frame = frame.loc[frame["join_game_name_key"].notna()].copy()
    invalid_key_count = before_count - len(frame)
    if frame.empty:
        raise ValueError("No valid canonical game keys remain after preprocessing.")
    flags = ["has_sales_match", "has_rental_match", "has_eilers_ts_match"]
    for name in flags:
        frame[name] = frame[name].map(as_bool)
    frame["parent_own_status"] = frame["parent_own_status"].map(normalize_game_reference).fillna("__unknown__")
    frame["_document_tiebreak"] = [
        hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()
        for record in json.loads(frame.to_json(orient="records", date_format="iso"))
    ]
    frame = frame.sort_values(["game_name", "_document_tiebreak"], kind="stable", na_position="last")
    mappings = list(zip(flags, ["to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match"]))
    merged = []
    conflicting_mappings = 0
    for _, group in frame.groupby("join_game_name_key", sort=True):
        row = group.iloc[0].drop(labels=["_document_tiebreak"]).copy()
        if pd.isna(pd.to_datetime(row["game_release_date"], errors="coerce")):
            dates = pd.to_datetime(group["game_release_date"], errors="coerce").dropna()
            if not dates.empty:
                row["game_release_date"] = dates.min()
        parent_names = group["parent_game"].map(normalize_game_reference).dropna()
        row["parent_game"] = None if parent_names.empty else parent_names.iloc[0]
        for flag, name in mappings:
            row[flag] = bool(group[flag].any())
            usable = group.loc[group[flag], name].map(normalize_game_reference).notna()
            names = group.loc[group[flag], name].loc[usable]
            row[name] = None if names.empty else str(names.iloc[0]).strip()
            conflicting_mappings += int(names.map(normalize_game_reference).nunique() > 1)
        merged.append(row)
    frame = pd.DataFrame(merged).reset_index(drop=True)
    numeric_columns = list(dict.fromkeys(c for fields in feature_groups.values() for c in fields))
    release_age = pd.to_numeric(frame["max_months_since_release"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    missing_numeric_counts = {}
    for column in numeric_columns:
        numeric_values = pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
        missing_numeric_counts[column] = int(numeric_values.isna().sum())
        frame[column] = numeric_values.fillna(0.0)
    frame["max_months_since_release"] = release_age
    frame["game_release_date"] = pd.to_datetime(frame["game_release_date"], errors="coerce")
    records = frame.to_dict("records")
    names = frame["join_game_name_key"].astype(str).to_numpy()
    flags_by_source = {source: frame[flag].to_numpy(dtype=bool) for source, (flag, _) in zip(["sales", "rental", "eilers"], mappings)}
    mapping_columns = {"sales": "to_sales_name_match", "rental": "to_rental_name_match", "eilers": "to_eilers_ts_name_match"}
    valid_names = {source: frame[column].map(normalize_game_reference).notna().to_numpy(dtype=bool)
                   for source, column in mapping_columns.items()}
    parent_keys = frame["parent_game"].map(normalize_game_reference).to_numpy()
    canonical_lookup = {name: index for index, name in enumerate(names)}
    aliases = {}
    for index, record in enumerate(records):
        for column in mapping_columns.values():
            alias = normalize_game_reference(record[column])
            if alias is not None:
                aliases.setdefault(alias, set()).add(index)
    resolutions = []
    for target_index, parent_key in enumerate(parent_keys):
        if parent_key is None:
            resolutions.append((None, "no_parent"))
            continue
        matches = {canonical_lookup[parent_key]} if parent_key in canonical_lookup else aliases.get(parent_key, set())
        if len(matches) != 1:
            resolutions.append((None, "ambiguous_name" if matches else "not_found"))
            continue
        parent_index = next(iter(matches))
        resolutions.append((None, "self_reference") if parent_index == target_index else (int(parent_index), "resolved"))
    return {
        "frame": frame, "records": records, "game_names": names,
        "feature_groups": copy.deepcopy(feature_groups), "numeric_columns": numeric_columns,
        "release_dates": frame["game_release_date"].to_numpy(dtype="datetime64[ns]"),
        "flags": flags_by_source, "valid_names": valid_names, "mapping_columns": mapping_columns,
        "parent_keys": parent_keys, "has_parent": np.array([key is not None for key in parent_keys], dtype=bool),
        "parent_resolutions": resolutions,
        "preprocessing": {"input_rows": before_count, "invalid_key_rows": invalid_key_count,
                          "canonical_games": len(frame), "duplicates_merged": before_count - invalid_key_count - len(frame),
                          "conflicting_mapping_groups": conflicting_mappings,
                          "missing_or_nonfinite_numeric_by_feature": missing_numeric_counts,
                          "missing_or_nonfinite_numeric_values": sum(missing_numeric_counts.values()),
                          "deduplication": "canonical key; alphabetical document then content hash; merge source flags/mappings",
                          "numeric_missing_value": 0.0},
    }


def weighted_distances(target_vector, candidate_vectors, method):
    """Full-vector distance with deterministic zero-vector cosine behavior."""
    if method == "euclidean":
        return np.linalg.norm(candidate_vectors - target_vector, axis=1)
    if method != "cosine":
        raise ValueError("Distance method must be cosine or euclidean.")
    target_norm = np.linalg.norm(target_vector)
    norms = np.linalg.norm(candidate_vectors, axis=1)
    similarities = np.zeros(len(candidate_vectors), dtype=np.float64)
    np.divide(np.einsum("ij,j->i", candidate_vectors, target_vector), target_norm * norms,
              out=similarities, where=target_norm * norms > 0)
    similarities[(target_norm == 0) & (norms == 0)] = 1.0
    return np.clip(1.0 - np.clip(similarities, -1.0, 1.0), 0.0, 1.0)


def prepare_games_from_pandas(frame, feature_groups):
    """Prepare an already projected pandas feature frame without requiring Spark."""
    return prepare_game_frame(frame, feature_groups)


def similarity_output_schema():
    from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DoubleType, ArrayType, MapType, DateType
    neighbor = StructType([StructField("join_game_name_key", StringType(), False),
                           StructField("matched_source_name", StringType(), False),
                           StructField("similarity_score", DoubleType(), False), StructField("distance", DoubleType(), False)])
    fields = [StructField(name, StringType(), True) for name in [
        "join_game_name", "parent_game", "parent_game_process", "parent_game_match_status", "resolved_parent_game_name",
        "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match", "sales_rental_match_type",
        "game_category", "own_status", "parent_own_status", "game_classification", "experiment_readable_name",
        "experiment_id", "distance_method", "forecast_eligibility_status", "prediction_date_source"]]
    fields += [StructField(name, BooleanType(), False) for name in [
        "parent_in_sales_neighbors", "parent_in_rental_neighbors", "has_sales_match", "has_rental_match", "has_eilers_ts_match"]]
    fields += [StructField("max_months_since_release", DoubleType(), True),
               StructField("game_release_date", DateType(), True), StructField("prediction_date", DateType(), True),
               StructField("feature_group_weights", MapType(StringType(), DoubleType()), False)]
    fields += [StructField(source + "_similar_games", ArrayType(neighbor), False) for source in ["sales", "rental", "eilers"]]
    comparable = StructType([neighbor["join_game_name_key"], neighbor["similarity_score"], neighbor["distance"]])
    fields.append(StructField("overall_similar_games", ArrayType(comparable), False))
    commercial = StructType([StructField(channel, ArrayType(neighbor), False) for channel in ("sales", "rental")])
    fields.append(StructField("commercial_neighbor_candidates", commercial, False))
    fields.append(StructField("neighbor_history_threshold_months",
                              MapType(StringType(), MapType(StringType(), DateType())), False))
    return StructType(fields)


def generate_neighbors(prepared, settings, history_dates, selection_settings):
    """Compute one experiment using reusable prepared games and local caches.

    Required settings: raw_weights (or feature_group_weights), distance_method,
    parent_game_process. Other settings use the original notebook defaults.
    experiment_id/name are row metadata, excluded from semantic_config.
    """
    groups = prepared["feature_groups"]
    weights = copy.deepcopy(settings.get("raw_weights", settings.get("feature_group_weights", {})))
    if set(weights) - set(groups):
        raise ValueError(f"Unknown feature groups: {sorted(set(weights) - set(groups))}")
    if any(isinstance(v, bool) or not isinstance(v, Real) or not math.isfinite(float(v)) or v < 0 for v in weights.values()):
        raise ValueError("Feature weights must be finite, nonnegative real numbers.")
    active_groups = {group: columns for group, columns in groups.items() if columns and weights.get(group, 0) > 0}
    total = sum(float(weights[group]) for group in active_groups)
    if not math.isfinite(total) or total <= 0:
        raise ValueError("At least one feature group must have a positive weight.")
    normalized = {group: float(weights[group]) / total if group in active_groups else 0.0 for group in groups}
    method = str(settings.get("distance_method", "cosine")).strip().lower()
    process = str(settings.get("parent_game_process", "p2_current")).strip().lower()
    if method not in {"cosine", "euclidean"}:
        raise ValueError("distance_method must be cosine or euclidean.")
    if process not in {"p1_parent_games_only", "p2_current", "p3_include_parent"}:
        raise ValueError("Unrecognized parent_game_process.")
    limits = {"sales": 5, "rental": 5, "eilers": 5, **settings.get("neighbor_limits", {})}
    gap = settings.get("minimum_age_gap_months", 12)
    if set(limits) != {"sales", "rental", "eilers"}:
        raise ValueError("neighbor_limits must contain sales, rental and eilers only.")
    for name, value in {**limits, "minimum_age_gap_months": gap}.items():
        if isinstance(value, bool) or not isinstance(value, Integral) or value < 0:
            raise ValueError(f"{name} must be a nonnegative integer.")
    cutoff_date = pd.Timestamp(settings.get("neighbor_released_after", "2021-06-01"))
    if pd.isna(cutoff_date):
        raise ValueError("neighbor_released_after must be a valid date.")
    configured_date = settings.get("prediction_as_of_date")
    fallback_source = settings.get("prediction_as_of_source", "configured" if configured_date else "assumed_recent")
    fallback_date = pd.Timestamp(configured_date or datetime.now(timezone.utc).date())
    if pd.isna(fallback_date) or fallback_date.tzinfo is not None or cutoff_date.tzinfo is not None:
        raise ValueError("The planned prediction date and neighbor cutoff must be timezone-free calendar dates.")
    columns = list(dict.fromkeys(column for fields in active_groups.values() for column in fields))
    raw = prepared["frame"][columns].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float64)
    multipliers = np.ones(len(columns), dtype=np.float64)
    indices = {column: index for index, column in enumerate(columns)}
    for group, fields in active_groups.items():
        multipliers[[indices[column] for column in fields]] = math.sqrt(normalized[group] / len(fields))
    release_dates = prepared["release_dates"]
    undated = np.isnat(release_dates)
    as_of = np.datetime64(fallback_date.date())
    prediction_dates = np.where(undated, as_of, release_dates)
    names, records = prepared["game_names"], prepared["records"]
    flags, valid = prepared["flags"], prepared["valid_names"]
    usable = {source: flags[source] & valid[source] for source in limits}
    for channel in ("sales", "rental"):
        usable[channel] = usable[channel] & usable["eilers"]
    indices = np.arange(len(records))
    game_index = {name: index for index, name in enumerate(names)}

    def weighted_matrix(population):
        lower, upper = raw[population].min(axis=0), raw[population].max(axis=0)
        scaled = np.zeros_like(raw)
        np.divide(raw - lower, upper - lower, out=scaled, where=upper != lower)
        return np.clip(scaled, 0.0, 1.0) * multipliers, lower, upper

    # Descriptive comparisons can use the full catalog, even for its oldest
    # game. Forecast lists below have separate as-of scaling and eligibility.
    catalog_matrix, _, _ = weighted_matrix(indices)
    historical_scalers = {}
    last_key, last_matrix = None, None
    rows = []
    zero_vector_count = 0
    for target_index in np.lexsort((names, prediction_dates)):
        target_index = int(target_index)
        target, prediction_date = records[target_index], prediction_dates[target_index]
        others = indices[indices != target_index]
        content_distance = weighted_distances(catalog_matrix[target_index], catalog_matrix[others], method)
        order = np.lexsort((names[others], content_distance))[:5]
        overall = [{"join_game_name_key": str(names[others[i]]), "distance": float(content_distance[i]),
                    "similarity_score": max(0.0, 1.0 - float(content_distance[i]))} for i in order]
        lists = {source: [] for source in limits}
        threshold_months = {"sales": {}, "rental": {}}
        prediction_month = prediction_date.astype("datetime64[M]").astype("datetime64[D]")
        commercial_candidates = {"sales": [], "rental": []}
        # An undated game is assumed recent. It can be considered at/after
        # the pinned as-of date. Commercial history must precede this target month.
        historical = (~undated & (release_dates < prediction_date)) | (undated & (as_of <= prediction_date))
        fit_mask = historical.copy()
        fit_mask[target_index] = False
        status = "no_historical_scaling_population"
        if fit_mask.any():
            # Undated peers share the same date. Excluding self from the
            # scaling population can give each such target different bounds.
            key = pd.Timestamp(prediction_date).date().isoformat()
            if historical[target_index]:
                key = f"{key}/{names[target_index]}"
            if last_key != key:
                last_matrix, lower, upper = weighted_matrix(fit_mask)
                last_key = key
                historical_scalers[key] = {"fit_rule": "available peers at prediction date, excluding target",
                                           "fit_game_count": int(fit_mask.sum()),
                                           "feature_min": dict(zip(columns, lower.tolist())),
                                           "feature_max": dict(zip(columns, upper.tolist()))}
            zero_vector_count += int(np.linalg.norm(last_matrix[target_index]) == 0)
            distances = weighted_distances(last_matrix[target_index], last_matrix, method)
            base = fit_mask & (undated | (release_dates > np.datetime64(cutoff_date.date())))
            if not undated[target_index]:
                latest_neighbor = np.datetime64(pd.Timestamp(prediction_date) - pd.DateOffset(months=int(gap)))
                base &= undated | (release_dates <= latest_neighbor)
            if process == "p1_parent_games_only" and prepared["has_parent"][target_index]:
                base &= prepared["has_parent"]
            parent_index, _ = prepared["parent_resolutions"][target_index]
            for source, limit in limits.items():
                source_eligible = usable[source]
                if source in commercial_candidates:
                    source_eligible = (source_eligible & (history_dates[source] < prediction_month)
                                       & (history_dates["eilers"] < prediction_month))
                candidates = np.flatnonzero(base & source_eligible)
                ranked = list(candidates[np.lexsort((names[candidates], distances[candidates]))])
                if (process == "p3_include_parent" and source in {"sales", "rental"}
                    and parent_index is not None and fit_mask[parent_index] and source_eligible[parent_index]):
                    ranked = [parent_index] + [index for index in ranked if index != parent_index]
                seen_mappings = set()
                seen_pairs = set()
                name_column = prepared["mapping_columns"][source]
                target_mapping = normalize_game_reference(target.get(name_column))
                if source == "eilers" and target_mapping is None:
                    target_mapping = str(names[target_index])
                for candidate_index in ranked:
                    if limit == 0 or (source == "eilers" and len(lists[source]) >= limit):
                        break
                    matched_name = str(records[candidate_index][name_column]).strip()
                    mapping = normalize_game_reference(matched_name)
                    if mapping is None or mapping == target_mapping:
                        continue
                    is_commercial = source in commercial_candidates
                    if is_commercial:
                        eilers_key = normalize_game_reference(records[candidate_index]["to_eilers_ts_name_match"])
                        pair = (mapping, eilers_key)
                        if pair in seen_pairs:
                            continue
                        seen_pairs.add(pair)
                    elif mapping in seen_mappings:
                        continue
                    distance = float(distances[candidate_index])
                    entry = {"join_game_name_key": str(names[candidate_index]), "matched_source_name": matched_name,
                             "similarity_score": max(0.0, 1.0 - distance), "distance": distance}
                    if is_commercial:
                        commercial_candidates[source].append(entry)
                    if mapping not in seen_mappings and len(lists[source]) < limit:
                        lists[source].append(entry)
                        seen_mappings.add(mapping)
                # P3 reserves a shortlist slot. The saved list is then ordered by
                # distance and uses the nearest eligible games for minimum fallback.
                shortlist = sorted(lists[source], key=lambda item: (
                    item["distance"], item["join_game_name_key"], normalize_game_reference(item["matched_source_name"])))
                lists[source] = [item for rank, item in enumerate(shortlist, start=1)
                                 if item["distance"] <= selection_settings["distance_cutoff"]
                                 or rank <= selection_settings["minimum_neighbors"]]
                if source in threshold_months:
                    for item in lists[source]:
                        key = item["join_game_name_key"]
                        threshold_months[source][key] = history_dates[source][game_index[key]].item()
            status = "ready" if any(lists.values()) else "no_eligible_neighbors"
        parent_index, parent_status = prepared["parent_resolutions"][target_index]
        row = {column: optional_string(target, column) for column in [
            "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match", "game_category",
            "own_status", "parent_own_status", "game_classification"]}
        sale, rent = bool(flags["sales"][target_index]), bool(flags["rental"][target_index])
        age = target.get("max_months_since_release")
        row.update({
            "join_game_name": str(names[target_index]), "parent_game": prepared["parent_keys"][target_index],
            "parent_game_process": process, "parent_game_match_status": parent_status,
            "resolved_parent_game_name": None if parent_index is None else str(names[parent_index]),
            "parent_in_sales_neighbors": parent_index is not None and any(item["join_game_name_key"] == names[parent_index] for item in lists["sales"]),
            "parent_in_rental_neighbors": parent_index is not None and any(item["join_game_name_key"] == names[parent_index] for item in lists["rental"]),
            "has_sales_match": sale, "has_rental_match": rent, "has_eilers_ts_match": bool(flags["eilers"][target_index]),
            "sales_rental_match_type": "both" if sale and rent else "sales" if sale else "rental" if rent else None,
            "max_months_since_release": None if age is None or pd.isna(age) else float(age),
            "overall_similar_games": overall, "commercial_neighbor_candidates": commercial_candidates,
            "neighbor_history_threshold_months": threshold_months,
            "sales_similar_games": lists["sales"], "rental_similar_games": lists["rental"], "eilers_similar_games": lists["eilers"],
            "experiment_readable_name": settings.get("experiment_readable_name", settings.get("user_experiment_name", "")),
            "experiment_id": settings.get("experiment_id"), "distance_method": method,
            "game_release_date": None if np.isnat(release_dates[target_index]) else pd.Timestamp(release_dates[target_index]).date(),
            "prediction_date": pd.Timestamp(prediction_date).date(),
            "prediction_date_source": "game_release_date" if not undated[target_index] else (
                "assumed_recent" if fallback_source == "assumed_recent" else "prediction_as_of_date"),
            "forecast_eligibility_status": status, "feature_group_weights": normalized.copy(),
        })
        rows.append(row)
    rows.sort(key=lambda row: row["join_game_name"])
    config = {
        "normalized_weights": normalized, "distance_method": method, "parent_game_process": process,
        "neighbor_limits": {source: int(limits[source]) for source in sorted(limits)},
        "neighbor_released_after": cutoff_date.date().isoformat(), "minimum_age_gap_months": int(gap),
        "prediction_as_of_date": None if pd.isna(fallback_date) else fallback_date.date().isoformat(),
        "feature_groups": copy.deepcopy(groups), "active_feature_groups": copy.deepcopy(active_groups), "feature_columns": columns,
        "scaling": {"method": "per_target_historical_minmax", "fit_population": "earlier known releases plus undated peers at/after the assumed date; exclude self",
                    "bounds": [0.0, 1.0], "clip_out_of_range": True, "constant_feature_transform": 0.0,
                    "numeric_missing_value": 0.0, "group_feature_multiplier": "sqrt(normalized_group_weight / feature_count)"},
        "eligibility": {"target": "all valid canonical source game keys, independent of actuals/matches",
                        "overall": "top five other catalog games by weighted content distance; descriptive only",
                        "normal_neighbor": "sales requires sales and Eilers; rental requires rental and Eilers; performance requires Eilers",
                        "dates": "known pairs obey release order, gap and cutoff; waive gap for undated targets or peers; undated peers available from the pinned date",
                        "grouping_restrictions": [], "p1": "target with a parent requires candidates with parents",
                        "p3": "commercial parent first; may override age gap/cutoff only; channel mappings and temporal availability required",
                        "commercial_candidate_order": "history-qualified ranked channel/Eilers pairs; final lists apply deduplication, top N and cutoff/fallback",
                        "mapping_deduplication": "normalized unique source mappings; exclude target mapping before top N"},
        "deduplication": prepared["preprocessing"]["deduplication"], "tie_breaker": "distance ascending, canonical key ascending",
    }
    counts = {"target_count": len(rows),
              "targets_with_overall_neighbors": sum(bool(row["overall_similar_games"]) for row in rows),
              "undated_targets": int(undated.sum()), "targets_with_sales_neighbors": sum(bool(row["sales_similar_games"]) for row in rows),
              "targets_with_rental_neighbors": sum(bool(row["rental_similar_games"]) for row in rows),
              "targets_with_eilers_neighbors": sum(bool(row["eilers_similar_games"]) for row in rows),
              "targets_without_source_matches": sum(not row["has_sales_match"] and not row["has_rental_match"] and not row["has_eilers_ts_match"] for row in rows),
              "target_zero_vector_count": zero_vector_count}
    for source in limits:
        neighbor_counts = [len(row[source + "_similar_games"]) for row in rows]
        counts[f"targets_without_{source}_neighbors"] = sum(count == 0 for count in neighbor_counts)
        counts[f"mean_{source}_neighbors"] = sum(neighbor_counts) / len(neighbor_counts) if neighbor_counts else 0.0
    return {"rows": rows, "schema": similarity_output_schema(), "semantic_config": config,
            "preprocessing": copy.deepcopy(prepared["preprocessing"]), "counts": counts,
            "diagnostics": {"forecast_eligibility_status": dict(Counter(row["forecast_eligibility_status"] for row in rows)),
                            "parent_resolution_status": dict(Counter(row["parent_game_match_status"] for row in rows))},
            "scalers": historical_scalers,
            "input_json": prepared["frame"].to_json(orient="records", date_format="iso", double_precision=15)}


def _functions():
    from pyspark.sql import functions as F
    return F


def _normalized(column):
    F = _functions()
    return F.regexp_replace(F.lower(F.trim(column.cast("string"))), r"\s+", " ")


def _valid_name(column):
    value = _normalized(column)
    return value.isNotNull() & ~value.isin("", "none", "null", "nan", "na", "n/a", "<na>")


def _finite(column):
    F = _functions()
    value = column.cast("double")
    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value)


def _require_columns(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")


def monthly_source_frames(source_frames, values):
    """Build one row per commercial game-month and per Eilers game-month/slot bucket."""
    F = _functions()
    missing = {"sales", "lease", "performance"} - set(source_frames)
    if missing:
        raise ValueError(f"Missing source frames: {sorted(missing)}")
    sales, lease, perf = (source_frames[name] for name in ["sales", "lease", "performance"])
    _require_columns(sales, ["normalized_game_name", "revenue_type", "beginning_month_date", "expr1_sum"], "Sales")
    _require_columns(lease, ["theme_name_eilers", "month", values["lease_value_column"]], "Rental")
    _require_columns(perf, ["game_name", "own_status", "yearmonth", "no_of_slots", *values["performance_metrics"]], "Eilers")
    sales_monthly = (
        sales.where(_normalized(F.col("revenue_type")).isin("sale", "sales"))
        .where(_valid_name(F.col("normalized_game_name")))
        .select(_normalized(F.col("normalized_game_name")).alias("_source_key"), F.lit("sales").alias("revenue_type"),
                F.trunc(F.to_date("beginning_month_date"), "month").alias("_calendar_month"),
                _finite(F.col("expr1_sum")).alias("_monthly_value"))
    )
    lease_monthly = (
        lease.where(_valid_name(F.col("theme_name_eilers")))
        .select(_normalized(F.col("theme_name_eilers")).alias("_source_key"), F.lit("lease").alias("revenue_type"),
                F.trunc(F.to_date("month"), "month").alias("_calendar_month"),
                _finite(F.col(values["lease_value_column"])).alias("_monthly_value"))
    )
    commercial = (
        sales_monthly.unionByName(lease_monthly)
        .where(F.col("_calendar_month").isNotNull() & F.col("_monthly_value").isNotNull())
        .groupBy("revenue_type", "_source_key", "_calendar_month")
        .agg(F.sum("_monthly_value").alias("_monthly_value"))
    )
    performance = (
        perf.select(_normalized(F.col("game_name")).alias("_game_key"),
                    _normalized(F.col("own_status")).alias("own_status"),
                    F.trunc(F.to_date(F.col("yearmonth").cast("string"), "yyyy-MM-dd"), "month").alias("_perf_month"),
                    _finite(F.col("no_of_slots")).alias("no_of_slots"),
                    *[_finite(F.col(metric)).alias(metric) for metric in values["performance_metrics"]])
        .where(_valid_name(F.col("_game_key")) & F.col("own_status").isin("owned", "leased")
               & F.col("_perf_month").isNotNull() & (F.col("no_of_slots") > 0))
    )
    if values["slot_counts"]:
        performance = performance.where(F.col("no_of_slots").isin(values["slot_counts"]))
    # Original 06 averages duplicates per game/month/slot bucket for neighbors.
    # This aggregation is independent of experiment/target and can be shared.
    performance_monthly = performance.groupBy("_game_key", "own_status", "_perf_month", "no_of_slots").agg(
        *[F.avg(metric).alias(metric) for metric in values["performance_metrics"]])
    return commercial, performance_monthly


def commercial_history_thresholds(commercial, sales_months, rental_months):
    """Find the first month each source name reaches its channel minimum.

    The input already has one finite observed value per channel, name and month.
    Comparing this one date per source with each target's prediction month is
    equivalent to counting that target's distinct, strictly earlier months.
    """
    ordered = commercial.selectExpr(
        "revenue_type", "_source_key", "_calendar_month",
        "row_number() over (partition by revenue_type, _source_key "
        "order by _calendar_month) as observed_month_number",
    )
    return (
        ordered.where(
            f"(revenue_type = 'sales' AND observed_month_number = {sales_months}) OR "
            f"(revenue_type = 'lease' AND observed_month_number = {rental_months})"
        )
        .selectExpr("revenue_type", "_source_key", "_calendar_month as history_threshold_month",
                    "observed_month_number as minimum_observed_months_required")
    )


def prepare_neighbor_history(games, source_frames, settings):
    """Align one commercial threshold date and one Eilers availability date per mapped game."""
    F = _functions()
    commercial, performance = monthly_source_frames(source_frames, settings)
    thresholds = commercial_history_thresholds(
        commercial, settings["sales_min_observed_months"], settings["rental_min_observed_months"])
    metric_present = F.coalesce(*[F.col(metric) for metric in settings["performance_metrics"]]).isNotNull()
    first_eilers = (performance.where(metric_present).groupBy("_game_key")
                    .agg(F.min("_perf_month").alias("first_month")))
    dates = {"sales": {}, "rental": {}, "eilers": {}}
    for row in thresholds.toLocalIterator():
        channel = "sales" if row["revenue_type"] == "sales" else "rental"
        dates[channel][row["_source_key"]] = row["history_threshold_month"]
    for row in first_eilers.toLocalIterator():
        dates["eilers"][row["_game_key"]] = row["first_month"]
    return align_history_dates(games, dates)


def align_history_dates(games, dates):
    aligned = {}
    for channel, column in games["mapping_columns"].items():
        aligned[channel] = np.array(
            [dates[channel].get(normalize_game_reference(record[column])) for record in games["records"]],
            dtype="datetime64[D]",
        )
    return aligned


## Durable results, attempts, and repairable tracking

Helper definitions only. Processing supports one active manual writer.


In [0]:
"""Durable similarity results for a single active, manually started writer.

Delta is authoritative. MLflow is retryable tracking, not the completion signal.
This module does not coordinate concurrent writers and performs no resets.
"""

from __future__ import annotations

import hashlib
import json
import math
import uuid
from datetime import date, datetime, timezone
from decimal import Decimal


RUN_SCHEMA = """
work_key string, experiment_id string, implementation_version string,
experiment_name string, config_json string, config_hash string,
source_path string, source_table_version long, source_versions_json string,
metadata_json string, status string, processing_status string,
tracking_status string, similarity_mlflow_run_id string,
attempt_id string, attempt_count long, started_at_utc string,
updated_at_utc string, completed_at_utc string,
error_type string, error_message string, tracking_error string,
result_table string, result_row_count long, result_hash string,
result_columns_json string, result_schema_json string, output_table_version long
"""
ATTEMPT_SCHEMA = """
attempt_id string, work_key string, experiment_id string, attempt_number long,
status string, started_at_utc string, completed_at_utc string,
error_type string, error_message string
"""
HASH_EXCLUDED = {"similarity_mlflow_run_id", "similarity_result_hash"}


def _now():
    return datetime.now(timezone.utc).isoformat()


def _normal(value):
    if hasattr(value, "asDict"):
        return _normal(value.asDict(recursive=True))
    if isinstance(value, dict):
        return {str(key): _normal(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_normal(item) for item in value]
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)
    if hasattr(value, "item"):
        return _normal(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError("Nonfinite values cannot be persisted as canonical metadata/results.")
    return value


def _json(value):
    return json.dumps(_normal(value), sort_keys=True, separators=(",", ":"), allow_nan=False)


def _quote(name):
    return ".".join("`" + part.replace("`", "``") + "`" for part in name.split("."))


def _signature(frame, columns):
    # The game dataset is deliberately bounded. Only generated result rows are
    # collected; this never collects the underlying sales/performance sources.
    digests = sorted(
        hashlib.sha256(_json(row).encode()).hexdigest()
        for row in frame.select(*columns).toLocalIterator()
    )
    return len(digests), hashlib.sha256("\n".join(digests).encode()).hexdigest()


def _find_mapping(value, name):
    if not isinstance(value, dict):
        return None
    if isinstance(value.get(name), dict):
        return value[name]
    for key in sorted(value):
        found = _find_mapping(value[key], name)
        if found is not None:
            return found
    return None


def _numeric_metrics(value, prefix=""):
    metrics = {}
    for key, item in value.items():
        name = f"{prefix}/{key}" if prefix else str(key)
        if isinstance(item, dict):
            metrics.update(_numeric_metrics(item, name))
        elif isinstance(item, (int, float)) and not isinstance(item, bool) and math.isfinite(float(item)):
            metrics[name] = float(item)
    return metrics


class SimilarityStore:
    """Store API used by 05's batch driver.

    ``begin(work_key, metadata)`` durably registers an attempt before compute;
    its returned ``skip_compute`` is true only for a verified completed run.
    ``publish(rows, schema, metadata)`` requires metadata['work_key'] and saves
    generated rows. Metadata can contain configuration/config_json, config_hash,
    experiment_name/label, source_path, source_table_version, source_versions,
    preprocessing, coverage and input_snapshot. Tracking artifacts are retained
    in the ledger so ``repair_tracking(work_key)`` needs no generation step.
    """

    def __init__(self, spark, result_table, runs_table, attempts_table,
                 mlflow_path, implementation_version):
        self.spark = spark
        self.result_table = result_table
        self.runs_table = runs_table
        self.attempts_table = attempts_table
        self.mlflow_path = mlflow_path
        self.implementation_version = implementation_version

    def initialize(self):
        for table, schema in [(self.runs_table, RUN_SCHEMA),
                              (self.attempts_table, ATTEMPT_SCHEMA)]:
            self.spark.sql(f"CREATE TABLE IF NOT EXISTS {_quote(table)} ({schema}) USING DELTA")
            required = set(self.spark.createDataFrame([], schema=schema).columns)
            missing = required - set(self.spark.table(table).columns)
            if missing:
                raise ValueError(f"Ledger {table} has an incompatible schema: missing {sorted(missing)}")
        return self

    def _merge(self, table, schema, key, record):
        frame = self.spark.createDataFrame([record], schema=schema)
        name = "similarity_store_" + uuid.uuid4().hex
        frame.createOrReplaceTempView(name)
        try:
            self.spark.sql(
                f"MERGE INTO {_quote(table)} AS target USING `{name}` AS source "
                f"ON target.`{key}` = source.`{key}` "
                "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"
            )
        finally:
            self.spark.catalog.dropTempView(name)

    def get_run(self, work_key):
        from pyspark.sql import functions as F

        if not self.spark.catalog.tableExists(self.runs_table):
            return None
        rows = (self.spark.table(self.runs_table).where(F.col("work_key") == work_key)
                .limit(2).collect())
        if len(rows) > 1:
            raise RuntimeError(f"Duplicate work records for {work_key}; stop the batch and resolve them.")
        return rows[0].asDict(recursive=True) if rows else None

    def _save_run(self, record):
        record["updated_at_utc"] = _now()
        record["status"] = record["processing_status"]
        self._merge(self.runs_table, RUN_SCHEMA, "work_key", record)
        return record

    def _metadata(self, metadata):
        metadata = _normal(dict(metadata))
        if metadata.get("config_json") is not None:
            config = json.loads(metadata["config_json"])
        else:
            config = metadata.get("configuration", metadata.get("config", {}))
        metadata["config_json"] = _json(config)
        metadata["config_hash"] = metadata.get("config_hash") or hashlib.sha256(
            metadata["config_json"].encode()).hexdigest()
        metadata["source_path"] = metadata.get("source_path") or metadata.get("source_table")
        if metadata.get("source_table_version") is not None:
            metadata["source_table_version"] = int(metadata["source_table_version"])
        metadata["source_versions"] = metadata.get("source_versions", {})
        metadata["experiment_name"] = str(
            metadata.get("experiment_name") or metadata.get("label") or "similarity")
        return metadata

    def _assert_identity(self, record, metadata):
        if record["implementation_version"] != self.implementation_version:
            raise ValueError("The work key is already assigned to a different implementation version.")
        for name in ["config_hash", "config_json", "source_path", "source_table_version"]:
            if record.get(name) != metadata.get(name):
                raise ValueError(f"The work key is already assigned to different {name}.")
        if record["source_versions_json"] != _json(metadata["source_versions"]):
            raise ValueError("The work key is already assigned to different source snapshots.")

    def _attempt(self, record, status, error=None):
        if not record.get("attempt_id"):
            return
        attempt = {
            "attempt_id": record["attempt_id"], "work_key": record["work_key"],
            "experiment_id": record["experiment_id"], "attempt_number": record["attempt_count"],
            "status": status, "started_at_utc": record["started_at_utc"],
            "completed_at_utc": (None if status == "running" else
                                 record.get("completed_at_utc") or _now()),
            "error_type": None if error is None else type(error).__name__,
            "error_message": None if error is None else str(error)[:16000],
        }
        self._merge(self.attempts_table, ATTEMPT_SCHEMA, "attempt_id", attempt)

    def begin(self, work_key, metadata):
        if not isinstance(work_key, str) or not work_key.strip():
            raise ValueError("A nonempty semantic work_key is required.")
        metadata = self._metadata({**metadata, "work_key": work_key})
        record = self.get_run(work_key)
        if record:
            self._assert_identity(record, metadata)
            if record["processing_status"] == "complete":
                self._verify(record)
                # Reconcile a crash between the authoritative completion commit
                # and the separate attempt-ledger commit.
                self._attempt(record, "complete")
                record = self.repair_tracking(work_key)
                return {**record, "skip_compute": True}
            if record["processing_status"] == "running":
                # The caller guarantees that the previous manual writer has
                # stopped. There is intentionally no time-based lease takeover.
                self._attempt(record, "interrupted")
            old_metadata = json.loads(record["metadata_json"])
            metadata = {**old_metadata, **metadata}
            metadata["requested_experiment_name"] = metadata["experiment_name"]
            # Names are audit labels, never new logical experiments. Retain
            # the first label so a renamed retry has identical result payload.
            metadata["experiment_name"] = record["experiment_name"]
        else:
            record = {
                "work_key": work_key,
                "experiment_id": "aiattr_" + hashlib.sha256(work_key.encode()).hexdigest()[:32],
                "implementation_version": self.implementation_version,
                "tracking_status": "pending", "similarity_mlflow_run_id": None,
                "attempt_count": 0, "completed_at_utc": None,
                "result_row_count": None, "result_hash": None,
                "result_columns_json": None, "result_schema_json": None,
                "output_table_version": None,
            }
        record.update(
            experiment_name=metadata["experiment_name"], config_json=metadata["config_json"],
            config_hash=metadata["config_hash"], source_path=metadata.get("source_path"),
            source_table_version=metadata.get("source_table_version"),
            source_versions_json=_json(metadata["source_versions"]), metadata_json=_json(metadata),
            processing_status="running", attempt_id="attempt_" + uuid.uuid4().hex,
            attempt_count=int(record["attempt_count"]) + 1, started_at_utc=_now(),
            error_type=None, error_message=None, tracking_error=None,
            result_table=self.result_table,
        )
        # Persist identity first, before computing or contacting MLflow.
        self._save_run(record)
        self._attempt(record, "running")
        record = self._ensure_tracking_run(record)
        return {**record, "skip_compute": False}

    def _ensure_tracking_run(self, record):
        try:
            from mlflow.tracking import MlflowClient
            from mlflow.entities import ViewType

            client = MlflowClient()
            if record.get("similarity_mlflow_run_id"):
                existing_run = client.get_run(record["similarity_mlflow_run_id"])
                if existing_run.info.lifecycle_stage != "active":
                    raise RuntimeError("The tracking run was deleted; restore it before repairing tracking.")
                return record
            experiment = client.get_experiment_by_name(self.mlflow_path)
            experiment_id = (experiment.experiment_id if experiment is not None
                             else client.create_experiment(self.mlflow_path))
            # Recover a run created immediately before a driver crash, before
            # its ID reached Delta. work_key is supplied by the driver as a
            # semantic key; experiment_id is a safe deterministic tag value.
            matches = client.search_runs(
                [experiment_id],
                filter_string=f"tags.similarity_experiment_id = '{record['experiment_id']}'",
                run_view_type=ViewType.ALL, max_results=2,
            )
            if len(matches) > 1:
                raise RuntimeError("Multiple MLflow runs own this similarity experiment identity.")
            if matches:
                if matches[0].info.lifecycle_stage != "active":
                    raise RuntimeError("The tracking run was deleted; restore it before repairing tracking.")
                run_id = matches[0].info.run_id
            else:
                run_id = client.create_run(experiment_id, tags={
                    "mlflow.runName": f"{record['experiment_name']}_{record['experiment_id'][-8:]}",
                    "pipeline_stage": "similarity", "pipeline_work_key": record["work_key"],
                    "similarity_experiment_id": record["experiment_id"],
                    "similarity_config_hash": record["config_hash"],
                    "implementation_version": self.implementation_version,
                }).info.run_id
            record.update(similarity_mlflow_run_id=run_id, tracking_status="pending", tracking_error=None)
        except Exception as error:
            record.update(tracking_status="failed", tracking_error=f"{type(error).__name__}: {error}"[:16000])
        return self._save_run(record)

    def _result_frame(self, record):
        from pyspark.sql import functions as F

        if not self.spark.catalog.tableExists(self.result_table):
            return None
        return self.spark.table(self.result_table).where(F.col("experiment_id") == record["experiment_id"])

    def _verify(self, record):
        from pyspark.sql import functions as F

        frame = self._result_frame(record)
        if frame is None or not record.get("result_columns_json"):
            raise RuntimeError("The completed run has no verifiable result output.")
        columns = json.loads(record["result_columns_json"])
        missing = set(columns) - set(frame.columns)
        if missing:
            raise RuntimeError(f"Saved results lost columns: {sorted(missing)}")
        count, content_hash = _signature(frame, columns)
        if count != record["result_row_count"] or content_hash != record["result_hash"]:
            raise RuntimeError("Saved result content/count differs from the durable work record.")
        bad_identity = frame.where(
            ~F.col("similarity_config_hash").eqNullSafe(F.lit(record["config_hash"]))
            | ~F.col("similarity_result_hash").eqNullSafe(F.lit(record["result_hash"]))
        ).limit(1).count()
        if bad_identity:
            raise RuntimeError("Saved result lineage differs from the durable work record.")
        return count

    def publish(self, rows, schema, metadata):
        from pyspark.sql import functions as F

        work_key = metadata.get("work_key")
        record = self.get_run(work_key)
        if record is None:
            raise ValueError("Call begin(work_key, metadata) before generating/publishing results.")
        combined_metadata = {**json.loads(record["metadata_json"]), **metadata}
        combined_metadata = self._metadata(combined_metadata)
        self._assert_identity(record, combined_metadata)
        combined_metadata["requested_experiment_name"] = combined_metadata["experiment_name"]
        combined_metadata["experiment_name"] = record["experiment_name"]
        if record["processing_status"] == "complete":
            self._verify(record)
            return self.repair_tracking(work_key)
        if record["processing_status"] != "running":
            raise ValueError("Call begin again to retry a failed/interrupted work item.")
        record["metadata_json"] = _json(combined_metadata)
        frame = self.spark.createDataFrame(rows, schema=schema)
        frame = (frame.withColumn("experiment_id", F.lit(record["experiment_id"]))
                 .withColumn("similarity_work_key", F.lit(work_key))
                 .withColumn("similarity_config_hash", F.lit(record["config_hash"]))
                 .withColumn("source_table_version", F.lit(record["source_table_version"]).cast("long")))
        configuration = json.loads(record["config_json"])
        revision = combined_metadata.get("data_revision", configuration.get("data_revision"))
        if revision is not None:
            frame = frame.withColumn("data_revision", F.lit(str(revision)))
        if "target_game_key" not in frame.columns and "join_game_name" in frame.columns:
            frame = frame.withColumn("target_game_key", F.regexp_replace(
                F.lower(F.trim(F.col("join_game_name"))), r"\s+", " "))
        if "experiment_name" not in frame.columns:
            frame = frame.withColumn("experiment_name", F.lit(record["experiment_name"]))
        if "experiment_readable_name" not in frame.columns:
            frame = frame.withColumn("experiment_readable_name", F.lit(record["experiment_name"]))
        columns = sorted(set(frame.columns) - HASH_EXCLUDED)
        row_count, content_hash = _signature(frame, columns)
        if record.get("result_hash") and (
            record["result_hash"] != content_hash or record["result_row_count"] != row_count
        ):
            raise RuntimeError("Retry produced different results for an existing immutable work key.")
        frame = (frame.withColumn("similarity_result_hash", F.lit(content_hash))
                 .withColumn("similarity_mlflow_run_id", F.lit(record.get("similarity_mlflow_run_id")).cast("string")))
        record.update(result_row_count=row_count, result_hash=content_hash,
                      result_columns_json=_json(columns), result_schema_json=frame.schema.json())
        # Save expected content and repair payload before the result commit.
        self._save_run(record)
        existing = self._result_frame(record)
        if existing is None or not existing.limit(1).count():
            (frame.write.format("delta").mode("append").option("mergeSchema", "true")
             .option("txnAppId", f"similarity:{record['experiment_id']}")
             .option("txnVersion", 0)
             .option("userMetadata", _json({"experiment_id": record["experiment_id"], "work_key": work_key}))
             .saveAsTable(self.result_table))
        self._verify(record)
        commit = (self.spark.sql(f"DESCRIBE HISTORY {_quote(self.result_table)}")
                  .where(F.get_json_object(F.col("userMetadata"), "$.experiment_id") == record["experiment_id"])
                  .agg(F.min("version").alias("v")).first()["v"])
        record.update(processing_status="complete", completed_at_utc=_now(),
                      output_table_version=None if commit is None else int(commit),
                      error_type=None, error_message=None)
        self._save_run(record)
        self._attempt(record, "complete")
        return self.repair_tracking(work_key)

    def repair_tracking(self, work_key):
        """Best-effort tracking repair; never recompute or change result content."""
        record = self.get_run(work_key)
        if record is None:
            raise ValueError(f"Unknown work key: {work_key}")
        if record["processing_status"] != "complete":
            return record
        if record["tracking_status"] == "complete":
            return record
        record = self._ensure_tracking_run(record)
        if not record.get("similarity_mlflow_run_id"):
            return record
        try:
            import mlflow
            from mlflow.tracking import MlflowClient
            from pyspark.sql import functions as F

            self._verify(record)
            client = MlflowClient()
            run_id = record["similarity_mlflow_run_id"]
            metadata = json.loads(record["metadata_json"])
            configuration = json.loads(record["config_json"])
            client.log_dict(run_id, configuration, "configuration.json")
            if metadata.get("config_text") is not None:
                client.log_text(run_id, metadata["config_text"], "configuration/batch.yaml")
            for key, artifact in [
                ("preprocessing", "preprocessing/snapshot.json"),
                ("coverage", "results/coverage.json"),
                ("input_snapshot", "data/prepared_games.json"),
            ]:
                if metadata.get(key) is not None:
                    payload = metadata[key]
                    client.log_dict(run_id, payload if isinstance(payload, dict) else {key: payload}, artifact)
            client.log_dict(run_id, {
                "work_key": work_key, "experiment_id": record["experiment_id"],
                "config_hash": record["config_hash"], "source_path": record["source_path"],
                "source_table_version": record["source_table_version"],
                "source_versions": json.loads(record["source_versions_json"]),
                "result_table": self.result_table, "row_count": record["result_row_count"],
                "result_hash": record["result_hash"], "delta_commit_version": record["output_table_version"],
                "implementation_version": self.implementation_version,
                "spark_version": self.spark.version, "mlflow_version": mlflow.__version__,
            }, "results/delta_location.json")
            settings = configuration.get("similarity", configuration)
            params = {
                "similarity_config_hash": record["config_hash"],
                "implementation_version": self.implementation_version,
                "source_table": record["source_path"],
                "source_table_version": record["source_table_version"],
                "data_revision": configuration.get("data_revision", metadata.get("data_revision")),
            }
            for key in ["distance_method", "parent_game_process", "minimum_age_gap_months",
                        "neighbor_released_after", "prediction_as_of_date"]:
                params[key] = settings.get(key, configuration.get(key))
            for group, value in (settings.get("raw_weights") or settings.get("feature_group_weights") or {}).items():
                params[f"raw_weight_{group}"] = value
            for source, value in settings.get("neighbor_limits", {}).items():
                params[f"n_{source}_neighbors"] = value
            normalized = _find_mapping(metadata.get("preprocessing", {}), "normalized_weights")
            if normalized is None and "feature_group_weights" in self._result_frame(record).columns:
                weight_row = self._result_frame(record).select("feature_group_weights").first()
                normalized = weight_row["feature_group_weights"] if weight_row else None
            for group, value in (normalized or {}).items():
                params[f"weight_{group}"] = value
            params = {key: str(value) for key, value in params.items() if value is not None}
            existing_params = client.get_run(run_id).data.params
            conflicts = [key for key, value in params.items()
                         if key in existing_params and existing_params[key] != value]
            if conflicts:
                raise RuntimeError(f"Immutable tracking parameters differ: {sorted(conflicts)}")
            from mlflow.entities import Param, Metric
            import time

            metrics = {"targets": float(record["result_row_count"]),
                       **_numeric_metrics(metadata.get("coverage", {}), "coverage")}
            client.log_batch(
                run_id,
                params=[Param(key, value) for key, value in params.items() if key not in existing_params],
                metrics=[Metric(key, value, int(time.time() * 1000), 0) for key, value in metrics.items()],
                synchronous=True,
            )
            client.set_tag(run_id, "neighbor_save_status", "complete")
            client.set_terminated(run_id, status="FINISHED")
            # This is lineage enrichment only. It is excluded from result hash.
            target_name = "similarity_tracking_" + uuid.uuid4().hex
            patch = self.spark.createDataFrame(
                [(record["experiment_id"], run_id)], "experiment_id string, run_id string")
            patch.createOrReplaceTempView(target_name)
            try:
                conflicting = self._result_frame(record).where(
                    F.col("similarity_mlflow_run_id").isNotNull()
                    & (F.col("similarity_mlflow_run_id") != run_id)
                ).limit(1).count()
                if conflicting:
                    raise RuntimeError("Results already reference a different MLflow run.")
                self.spark.sql(
                    f"MERGE INTO {_quote(self.result_table)} AS target USING `{target_name}` AS source "
                    "ON target.experiment_id = source.experiment_id "
                    "WHEN MATCHED AND target.similarity_mlflow_run_id IS NULL THEN "
                    "UPDATE SET target.similarity_mlflow_run_id = source.run_id"
                )
            finally:
                self.spark.catalog.dropTempView(target_name)
            self._verify(record)
            record.update(tracking_status="complete", tracking_error=None)
        except Exception as error:
            record.update(tracking_status="failed", tracking_error=f"{type(error).__name__}: {error}"[:16000])
        return self._save_run(record)

    def fail(self, work_key, error, attempt_id=None):
        record = self.get_run(work_key)
        if record is None:
            raise ValueError(f"Unknown work key: {work_key}")
        if attempt_id is not None and record["attempt_id"] != attempt_id:
            raise ValueError("Failure belongs to an earlier attempt; current work record is unchanged.")
        # A tracking failure must never downgrade verified published results.
        if record["processing_status"] == "complete":
            return record
        record.update(processing_status="failed", error_type=type(error).__name__,
                      error_message=str(error)[:16000])
        self._save_run(record)
        self._attempt(record, "failed", error)
        if record.get("similarity_mlflow_run_id"):
            try:
                from mlflow.tracking import MlflowClient

                client = MlflowClient()
                run_id = record["similarity_mlflow_run_id"]
                client.set_tag(run_id, "processing_status", "failed")
                client.set_tag(run_id, "failure_type", type(error).__name__)
                client.set_terminated(
                    run_id, status="KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            except Exception:
                # Preserve the compute failure and its committed Delta record.
                pass
        return record

    def status_df(self):
        return self.spark.table(self.runs_table)


## Load the YAML and plan the batch

Validates all configurations, checks migration state, and resolves the pinned source revision.


In [0]:
# Validate the entire YAML before creating status records or saving results.
from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql import functions as F

SHARED_FORECAST_CONFIG = load_forecast_config(CONFIG_PATH)
BATCH_CONFIG = SHARED_FORECAST_CONFIG["common"]
if not any(item["enabled"] for item in BATCH_CONFIG["experiments"]):
    raise ValueError("Enable at least one similarity experiment in the YAML.")
HISTORY_VERSIONS = resolve_forecast_source_versions(spark, SHARED_FORECAST_CONFIG)
SELECTION_HISTORY = neighbor_selection_policy(SHARED_FORECAST_CONFIG, HISTORY_VERSIONS)
preflight_similarity_tables(spark, BATCH_CONFIG)
RESOLVED_SOURCE_VERSION = resolve_similarity_source_version(spark, BATCH_CONFIG)
OUTPUTS = BATCH_CONFIG["outputs"]
# Pin the first batch date for undated targets in this revision. A retry or a
# new weight experiment using the same source snapshot reuses that date.
prediction_data = dict(BATCH_CONFIG["data"])
prediction_data["prediction_as_of_source"] = "configured"
if prediction_data["prediction_as_of_date"] is None:
    prediction_data["prediction_as_of_source"] = "assumed_recent"
    pins = []
    if spark.catalog.tableExists(OUTPUTS["runs_table"]):
        pins = (spark.table(OUTPUTS["runs_table"])
                .where((F.col("source_path") == prediction_data["ai_attributes_table"])
                       & (F.col("source_table_version") == RESOLVED_SOURCE_VERSION)
                       & (F.get_json_object("config_json", "$.data_revision") == prediction_data["revision"])
                       & (F.get_json_object("config_json", "$.prediction_as_of_source") == "assumed_recent"))
                .select(F.get_json_object("config_json", "$.prediction_as_of_date").alias("pinned_date"))
                .distinct().collect())
    if len(pins) > 1:
        raise ValueError("Multiple assumed prediction dates were saved for this AI revision and snapshot.")
    prediction_data["prediction_as_of_date"] = pins[0]["pinned_date"] if pins else datetime.now(timezone.utc).date().isoformat()
print("Prediction date for undated targets:", prediction_data["prediction_as_of_date"], prediction_data["prediction_as_of_source"])
STORE = SimilarityStore(
    spark, OUTPUTS["like_games_table"], OUTPUTS["runs_table"], OUTPUTS["attempts_table"],
    BATCH_CONFIG["tracking"]["experiment_path"], SIMILARITY_IMPLEMENTATION_VERSION,
).initialize()
BATCH_ID = "batch_" + uuid4().hex
BATCH_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
ENABLED_EXPERIMENTS = [item for item in BATCH_CONFIG["experiments"] if item["enabled"]]
PLANNED_EXPERIMENTS = []
seen_work_keys = set()
for item in ENABLED_EXPERIMENTS:
    work_key, semantic = similarity_work_identity(item, prediction_data, RESOLVED_SOURCE_VERSION, SELECTION_HISTORY)
    if work_key in seen_work_keys:
        continue
    seen_work_keys.add(work_key)
    PLANNED_EXPERIMENTS.append({"requested_name": item["name"], "work_key": work_key,
                                "settings": item["settings"], "semantic": semantic})
print("Selection rules and historical snapshots:", SELECTION_HISTORY)
print(f"Batch: {BATCH_ID}")
print(f"YAML: {BATCH_CONFIG['config_path']}")
print(f"Data revision: {BATCH_CONFIG['data']['revision']}; pinned AI source Delta version: {RESOLVED_SOURCE_VERSION}")
print(f"Enabled named experiments: {len(ENABLED_EXPERIMENTS)}. Distinct neighbor selections: {len(PLANNED_EXPERIMENTS)}. One active manual batch at a time.")
display(spark.createDataFrame([
    (item["requested_name"], item["work_key"], item["settings"]["parent_game_process"], item["settings"]["distance_method"])
    for item in PLANNED_EXPERIMENTS
], "experiment_name string, work_key string, parent_game_process string, distance_method string"))


## Process pending experiments

Completed work is verified and skipped. Independent failures are retained; other experiments continue. A failed batch raises after its summary is displayed.


In [0]:
# This cell can be retried after a failure, including in a fresh Python session.
# Rerun all cells if you edit the YAML or implementation.
if load_experiment_config(CONFIG_PATH) != BATCH_CONFIG:
    raise ValueError("The YAML changed after loading. Rerun the notebook from the top.")

import json
import time
from datetime import datetime, timezone

BATCH_OUTCOMES = []
PENDING_WORK = []


def batch_outcome(item, state=None, action="failed", error=None):
    state = state or {}
    metadata = json.loads(state.get("metadata_json") or "{}")
    coverage = metadata.get("coverage", {})
    counts = coverage.get("counts", coverage)
    return {
        "batch_id": BATCH_ID, "requested_name": item["requested_name"],
        "experiment_name": state.get("experiment_name", item["requested_name"]),
        "experiment_id": state.get("experiment_id"), "action": action,
        "processing_status": state.get("processing_status", "failed"),
        "tracking_status": state.get("tracking_status", "pending"),
        "attempt_count": state.get("attempt_count"), "target_count": state.get("result_row_count"),
        "with_sales_neighbors": counts.get("targets_with_sales_neighbors"),
        "with_rental_neighbors": counts.get("targets_with_rental_neighbors"),
        "with_eilers_neighbors": counts.get("targets_with_eilers_neighbors"),
        "error": None if error is None else f"{type(error).__name__}: {error}",
    }


# Register durable attempts first. A data revision is now pinned even if shared
# preparation or the driver fails. Completed work verifies its saved content and
# retries tracking, without generating neighbors again.
for item in PLANNED_EXPERIMENTS:
    metadata = {
        "work_key": item["work_key"], "configuration": item["semantic"],
        "config_hash": item["work_key"], "experiment_name": item["requested_name"],
        "data_revision": BATCH_CONFIG["data"]["revision"],
        "source_path": BATCH_CONFIG["data"]["ai_attributes_table"],
        "source_table_version": RESOLVED_SOURCE_VERSION,
        "source_versions": {"ai_attributes": RESOLVED_SOURCE_VERSION, **HISTORY_VERSIONS},
        "config_text": BATCH_CONFIG["config_text"], "yaml_sha256": BATCH_CONFIG["yaml_sha256"],
        "batch_id": BATCH_ID, "batch_started_at_utc": BATCH_STARTED_AT_UTC,
    }
    try:
        state = STORE.begin(item["work_key"], metadata)
        if state["skip_compute"]:
            BATCH_OUTCOMES.append(batch_outcome(item, state, "skipped_complete"))
        else:
            # Names are labels: an existing identity retains its original name
            # so a rename cannot change immutable result content on retry.
            metadata["experiment_name"] = state["experiment_name"]
            PENDING_WORK.append((item, metadata, state))
    except Exception as error:
        BATCH_OUTCOMES.append(batch_outcome(item, action="failed_to_begin", error=error))

prepared_games = None
if PENDING_WORK:
    preparation_started = time.monotonic()
    try:
        pinned_ai_source = spark.read.option("versionAsOf", RESOLVED_SOURCE_VERSION).table(
            BATCH_CONFIG["data"]["ai_attributes_table"])
        games = prepare_games(pinned_ai_source)
        history_frames = {key: spark.read.option("versionAsOf", HISTORY_VERSIONS[key]).table(path)
                          for key, path in SHARED_FORECAST_CONFIG["sources"].items()}
        history_dates = prepare_neighbor_history(games, history_frames, SELECTION_HISTORY["settings"])
        prepared_games = games
        preparation_seconds = time.monotonic() - preparation_started
        print(f"Prepared {len(prepared_games['frame'])} canonical games once for {len(PENDING_WORK)} pending experiments.")
    except BaseException as error:
        for item, metadata, state in PENDING_WORK:
            try:
                failed = STORE.fail(item["work_key"], error, state["attempt_id"])
                BATCH_OUTCOMES.append(batch_outcome(item, failed, "shared_preparation_failed", error))
            except Exception as status_error:
                BATCH_OUTCOMES.append(batch_outcome(item, state, "status_write_failed", status_error))
        if not isinstance(error, Exception):
            raise

if prepared_games is not None:
    for index, (item, metadata, state) in enumerate(PENDING_WORK, start=1):
        print(f"[{index}/{len(PENDING_WORK)}] {item['requested_name']} | {state['experiment_id']}")
        started = time.monotonic()
        try:
            settings = {
                **item["settings"], "experiment_id": state["experiment_id"],
                "experiment_readable_name": state["experiment_name"],
                "prediction_as_of_date": item["semantic"]["prediction_as_of_date"],
                "prediction_as_of_source": item["semantic"]["prediction_as_of_source"],
            }
            generation = generate_neighbors(prepared_games, settings, history_dates, SELECTION_HISTORY["settings"])
            metadata.update(
                preprocessing={**generation["preprocessing"], "algorithm": generation["semantic_config"],
                               "historical_scalers": generation["scalers"]},
                coverage={"counts": generation["counts"], "diagnostics": generation["diagnostics"]},
                input_snapshot={"games": json.loads(generation["input_json"])},
                shared_preparation_seconds=preparation_seconds,
                generation_seconds=time.monotonic() - started,
            )
            completed = STORE.publish(generation["rows"], generation["schema"], metadata)
            BATCH_OUTCOMES.append(batch_outcome(item, completed, "processed"))
            del generation
        except BaseException as error:
            try:
                failed = STORE.fail(item["work_key"], error, state["attempt_id"])
                BATCH_OUTCOMES.append(batch_outcome(item, failed, "failed", error))
            except Exception as status_error:
                BATCH_OUTCOMES.append(batch_outcome(item, state, "status_write_failed", status_error))
            if not isinstance(error, Exception):
                # On interruption, unvisited attempts stay visibly running and
                # are recovered by the next manually started batch.
                raise

import pandas as pd
batch_summary_pdf = pd.DataFrame(BATCH_OUTCOMES)
display(batch_summary_pdf)
failed_outcomes = [row for row in BATCH_OUTCOMES if row["error"] is not None]
tracking_warnings = [row for row in BATCH_OUTCOMES if row["tracking_status"] != "complete"]
print(f"Discovered={len(PLANNED_EXPERIMENTS)}, processed={sum(r['action'] == 'processed' for r in BATCH_OUTCOMES)}, "
      f"skipped={sum(r['action'] == 'skipped_complete' for r in BATCH_OUTCOMES)}, failed={len(failed_outcomes)}")
if tracking_warnings:
    print(f"Tracking needs attention for {len(tracking_warnings)} item(s). Verified Delta results remain complete; rerunning repairs tracking without recomputing neighbors.")
if failed_outcomes:
    raise RuntimeError("Some experiments failed. Their errors are shown above and retained in the status tables. Rerun to retry; completed work is skipped.")
if not PENDING_WORK:
    print("Nothing new to compute. Saved generations were verified.")


## Processing and coverage monitoring

Read-only history, status totals, and attempt details. This cell does not regenerate or reset results.


In [0]:
# Read-only monitoring. This cell can be run independently after loading setup.
# Do not infer ground-truth availability from mapping flags: 07 will determine
# evaluation eligibility from actual observed data, separately per channel/metric.
from pyspark.sql import functions as F

similarity_monitor_df = (
    STORE.status_df()
    .withColumn("data_revision", F.get_json_object("metadata_json", "$.data_revision"))
    .withColumn("duration_seconds", F.coalesce(F.col("completed_at_utc").cast("timestamp"),
                F.when(F.col("processing_status") == "running", F.current_timestamp())
                 .otherwise(F.col("updated_at_utc").cast("timestamp"))).cast("double")
                - F.col("started_at_utc").cast("timestamp").cast("double"))
    .withColumn("targets_with_sales_neighbors", F.get_json_object("metadata_json", "$.coverage.counts.targets_with_sales_neighbors").cast("long"))
    .withColumn("targets_with_rental_neighbors", F.get_json_object("metadata_json", "$.coverage.counts.targets_with_rental_neighbors").cast("long"))
    .withColumn("targets_with_eilers_neighbors", F.get_json_object("metadata_json", "$.coverage.counts.targets_with_eilers_neighbors").cast("long"))
    .withColumn("missing_prediction_date_targets", F.coalesce(F.get_json_object("metadata_json", "$.coverage.diagnostics.forecast_eligibility_status.missing_prediction_date").cast("long"), F.lit(0)))
    .withColumn("sales_neighbor_coverage", F.when(F.col("result_row_count") > 0, F.col("targets_with_sales_neighbors") / F.col("result_row_count")))
    .withColumn("rental_neighbor_coverage", F.when(F.col("result_row_count") > 0, F.col("targets_with_rental_neighbors") / F.col("result_row_count")))
    .withColumn("eilers_neighbor_coverage", F.when(F.col("result_row_count") > 0, F.col("targets_with_eilers_neighbors") / F.col("result_row_count")))
)
display(similarity_monitor_df.select(
    "experiment_name", "experiment_id", "data_revision", "source_table_version",
    "processing_status", "tracking_status", "attempt_count", "duration_seconds",
    "result_row_count", "targets_with_sales_neighbors", "targets_with_rental_neighbors", "targets_with_eilers_neighbors",
    "sales_neighbor_coverage", "rental_neighbor_coverage", "eilers_neighbor_coverage",
    "missing_prediction_date_targets", "error_message", "tracking_error", "similarity_mlflow_run_id",
    "updated_at_utc",
).orderBy(F.desc("updated_at_utc")).limit(BATCH_CONFIG["monitoring"]["history_rows"]))
display(STORE.status_df().groupBy("processing_status", "tracking_status").count())
display(spark.table(OUTPUTS["attempts_table"]).orderBy(F.desc("started_at_utc")).limit(
    BATCH_CONFIG["monitoring"]["history_rows"]))
print(f"Persistent status: {OUTPUTS['runs_table']}")
print(f"Persistent attempts: {OUTPUTS['attempts_table']}")
print("A completed save can have missing neighbors. Use the coverage and date reasons above to assess quality.")


In [0]:
from pyspark.sql import functions as F

p = OUTPUTS['like_games_table']
df = spark.table(p)
display(df)

In [0]:
from pyspark.sql import functions as F

df = spark.table(OUTPUTS["like_games_table"])

display(
    df.groupBy("experiment_id", "experiment_readable_name", "parent_game_process")
      .agg(
          F.count("*").alias("target_games"),
          F.count(F.when(F.size("overall_similar_games") > 0, 1)).alias("games_with_overall_neighbors"),
          F.count(F.when(F.col("prediction_date_source") == "assumed_recent", 1)).alias("assumed_recent_targets"),
          F.sum(
              F.when(F.size("sales_similar_games") > 0, 1).otherwise(0)
          ).alias("games_with_sales_neighbors"),
          F.sum(
              F.when(F.size("rental_similar_games") > 0, 1).otherwise(0)
          ).alias("games_with_rental_neighbors"),
          F.sum(
              F.when(F.size("eilers_similar_games") > 0, 1).otherwise(0)
          ).alias("games_with_eilers_neighbors"),
      )
      .orderBy("parent_game_process", "experiment_readable_name")
)

display(
    df.groupBy("parent_game_process", "forecast_eligibility_status")
      .count()
      .orderBy("parent_game_process", "forecast_eligibility_status")
)